# Train Multiple Versions on Kaggle & Track with W&B

**Task**: Fine-tune DistilBERT on AG News (4-class text classification)  
**Platform**: Kaggle (GPU T4 x2)  
**Tracking**: Weights & Biases (W&B)

**Prerequisites**:
- Enable GPU: Settings → Accelerator → GPU T4 x2
- Add Kaggle Secrets: `WANDB_API_KEY`, `HF_TOKEN`, `GITHUB_TOKEN`
- Dataset on HF Hub: [`Recurrent/prepared_data_mlops2`](https://huggingface.co/datasets/Recurrent/prepared_data_mlops2)

## Step 0: Install Dependencies

In [1]:
# !pip install -q transformers datasets wandb scikit-learn accelerate

In [2]:
print("Hello")

Hello


## Step 1: Load Secrets in Kaggle

API keys are stored securely using Kaggle Secrets — never hardcoded.

In [3]:
from kaggle_secrets import UserSecretsClient
import os
import wandb
from huggingface_hub import login

# Load secrets securely
secrets = UserSecretsClient()
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
HF_TOKEN = secrets.get_secret('HF_TOKEN')
login(token=HF_TOKEN)
wandb.login()

# GitHub token for pushing changes
github_token = secrets.get_secret('GITHUB_TOKEN')

print("Successfully authenticated with W&B, Hugging Face, and GitHub!")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


wandb: Currently logged in as: g25ait2087 (models-iit-jodhpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully authenticated with W&B, Hugging Face, and GitHub!


## Step 2: Load Prepared Dataset & Model

In [4]:
import json
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load dataset from Hugging Face Hub
HF_DATASET_REPO = "Recurrent/prepared_data_mlops2"
dataset = load_dataset(HF_DATASET_REPO, data_dir="prepared_data")
print(dataset)

# Label mapping
id2label = {
    "0": "World",
    "1": "Sports",
    "2": "Business",
    "3": "Sci/Tech"
}
label2id = {v: int(k) for k, v in id2label.items()}
num_labels = len(id2label)

print(f"\nLabels: {id2label}")
print(f"Number of classes: {num_labels}")

prepared_data/train_prepared.csv:   0%|          | 0.00/27.4M [00:00<?, ?B/s]

test_prepared.csv: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

Labels: {'0': 'World', '1': 'Sports', '2': 'Business', '3': 'Sci/Tech'}
Number of classes: 4


In [5]:
# Preview the data
print(f"Train samples: {len(dataset['train'])}")
print(f"Test samples:  {len(dataset['test'])}")
dataset["train"].to_pandas().head()

Train samples: 120000
Test samples:  7600


,text,label
0,explosion rocks baghdad neighborhood baghdad i...,0
1,bbc reporters log bbc correspondents record ev...,0
2,israel welcomes rice nomination palestinians w...,0
3,medical journal calls for a new drug watchdog ...,0
4,militants kidnap relatives of iraqi ministertv...,0


In [6]:
# Assign train/test splits
train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(f"Train dataset: {train_dataset}")
print(f"Test dataset:  {test_dataset}")

Train dataset: Dataset({
    features: ['text', 'label'],
    num_rows: 120000
})
Test dataset:  Dataset({
    features: ['text', 'label'],
    num_rows: 7600
})


In [7]:
# Load tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize datasets
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print("Tokenization complete!")
print(f"Sample features: {train_dataset[0].keys()}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Tokenization complete!
Sample features: dict_keys(['label', 'input_ids', 'attention_mask'])


## Step 3: Define Metrics

In [8]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }

---
## Step 4: Training Version 1

**Hyperparameters (v1)**:
- Learning rate: `2e-5`
- Batch size: `16`
- Epochs: `3`
- Weight decay: `0.01`

In [9]:
from transformers import TrainingArguments, Trainer

# Initialise W&B run for Version 1
wandb.init(
    project="mlops-assignment-group13",
    name="run-v1",
    config={
        "model": model_name,
        "epochs": 3,
        "batch_size": 16,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "max_length": 128,
        "version": "v1",
        "platform": "Kaggle",
    }
)

print("W&B run initialised: run-v1")

wandb: setting up run bfslsbr5


wandb: Tracking run with wandb version 0.25.1


wandb: Run data is saved locally in /kaggle/working/wandb/run-20260611_155924-bfslsbr5
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run run-v1


wandb: ⭐️ View project at https://wandb.ai/models-iit-jodhpur/mlops-assignment-group13


wandb: 🚀 View run at https://wandb.ai/models-iit-jodhpur/mlops-assignment-group13/runs/bfslsbr5


W&B run initialised: run-v1


In [10]:
# Load fresh model for v1
model_v1 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# Training arguments for v1
training_args_v1 = TrainingArguments(
    output_dir="./results-v1",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="wandb",
    run_name="run-v1",
    logging_steps=100,
)

# Create Trainer
trainer_v1 = Trainer(
    model=model_v1,
    args=training_args_v1,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# Train
print("Starting training v1...")
trainer_v1.train()

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training v1...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].


There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=11250, training_loss=0.3424789077758789, metrics={'train_runtime': 2682.4835, 'train_samples_per_second': 134.204, 'train_steps_per_second': 4.194, 'total_flos': 1.192249110528e+16, 'train_loss': 0.3424789077758789, 'epoch': 3.0})

In [11]:
# Evaluate v1
results_v1 = trainer_v1.evaluate()
print(f"\n=== Version 1 Results ===")
print(f"  Accuracy: {results_v1['eval_accuracy']:.4f}")
print(f"  F1 Score: {results_v1['eval_f1']:.4f}")
print(f"  Eval Loss: {results_v1['eval_loss']:.4f}")

# Finish W&B run
wandb.finish()
print("\nW&B run-v1 finished.")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


wandb: updating run metadata



=== Version 1 Results ===
  Accuracy: 0.9420
  F1 Score: 0.9420
  Eval Loss: 0.4221


wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml


wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁▅██
wandb:                 eval/f1 ▁▅██
wandb:               eval/loss ▄▁██
wandb:            eval/runtime ▂▇█▁
wandb: eval/samples_per_second ▇▂▁█
wandb:   eval/steps_per_second ▇▂▁█
wandb:             train/epoch ▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇█████
wandb:       train/global_step ▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇███
wandb:         train/grad_norm ▄▄▁▄▃▂█▂▇▃▂▂▂▄▅▄▄▁▅▂▃▄▄▃▂▄▃▁▂▇▃▃▄▅▄▅▂▁▁█
wandb:     train/learning_rate █████▇▇▇▇▇▆▆▆▆▆▆▆▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁
wandb:                      +1 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.94197
wandb:                 eval/f1 0.94196
wandb:               eval/loss 0.42209
wandb:            eval/runtime 16.2719
wandb: eval/samples_per_second 467.062
wandb:   eval/steps_per_second 7.313
wandb:              total_flos 1.192249110528e+16
wandb:             train/epoch 3
wandb:       train/global_step 11250
wandb:         train/grad_norm 17.81548
wandb:       

wandb: 🚀 View run run-v1 at: https://wandb.ai/models-iit-jodhpur/mlops-assignment-group13/runs/bfslsbr5
wandb: ⭐️ View project at: https://wandb.ai/models-iit-jodhpur/mlops-assignment-group13
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260611_155924-bfslsbr5/logs



W&B run-v1 finished.


---
## Step 5: Training Version 2

**Hyperparameters (v2)** — changed learning rate and batch size:
- Learning rate: `5e-5` (increased from 2e-5)
- Batch size: `32` (increased from 16)
- Epochs: `3`
- Weight decay: `0.01`

In [12]:
# Initialise W&B run for Version 2
wandb.init(
    project="mlops-assignment-group13",
    name="run-v2",
    config={
        "model": model_name,
        "epochs": 3,
        "batch_size": 32,
        "learning_rate": 5e-5,
        "weight_decay": 0.01,
        "max_length": 128,
        "version": "v2",
        "platform": "Kaggle",
    }
)

print("W&B run initialised: run-v2")

wandb: setting up run 6vbofn12


wandb: Tracking run with wandb version 0.25.1


wandb: Run data is saved locally in /kaggle/working/wandb/run-20260611_164432-6vbofn12
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run run-v2


wandb: ⭐️ View project at https://wandb.ai/models-iit-jodhpur/mlops-assignment-group13


wandb: 🚀 View run at https://wandb.ai/models-iit-jodhpur/mlops-assignment-group13/runs/6vbofn12


W&B run initialised: run-v2


In [13]:
# Load fresh model for v2 (start from pretrained, not from v1 checkpoint)
model_v2 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# Training arguments for v2
training_args_v2 = TrainingArguments(
    output_dir="./results-v2",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="wandb",
    run_name="run-v2",
    logging_steps=100,
)

# Create Trainer
trainer_v2 = Trainer(
    model=model_v2,
    args=training_args_v2,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# Train
print("Starting training v2...")
trainer_v2.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training v2...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.416185,0.384374,0.936842,0.936823
2,0.265048,0.368066,0.941184,0.941207
3,0.159596,0.422901,0.940921,0.940908


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].


There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=5625, training_loss=0.3057808285607232, metrics={'train_runtime': 2267.3694, 'train_samples_per_second': 158.774, 'train_steps_per_second': 2.481, 'total_flos': 1.192249110528e+16, 'train_loss': 0.3057808285607232, 'epoch': 3.0})

In [14]:
# Evaluate v2
results_v2 = trainer_v2.evaluate()
print(f"\n=== Version 2 Results ===")
print(f"  Accuracy: {results_v2['eval_accuracy']:.4f}")
print(f"  F1 Score: {results_v2['eval_f1']:.4f}")
print(f"  Eval Loss: {results_v2['eval_loss']:.4f}")

# Finish W&B run
wandb.finish()
print("\nW&B run-v2 finished.")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


wandb: updating run metadata



=== Version 2 Results ===
  Accuracy: 0.9412
  F1 Score: 0.9412
  Eval Loss: 0.3680


wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁███
wandb:                 eval/f1 ▁███
wandb:               eval/loss ▃▁█▁
wandb:            eval/runtime ▂▁█▅
wandb: eval/samples_per_second ▇█▁▄
wandb:   eval/steps_per_second ▇█▁▄
wandb:             train/epoch ▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇█████
wandb:       train/global_step ▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
wandb:         train/grad_norm ▃▄▄▁▄▃▂▄▁▃▄▂▂▄▃▂▁▂▂▄▇▄▂▂▂▃▂▅▂▂▁▅▂▄▇▄█▄▃▄
wandb:     train/learning_rate ████▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁
wandb:                      +1 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.94118
wandb:                 eval/f1 0.94121
wandb:               eval/loss 0.36804
wandb:            eval/runtime 16.277
wandb: eval/samples_per_second 466.916
wandb:   eval/steps_per_second 7.311
wandb:              total_flos 1.192249110528e+16
wandb:             train/epoch 3
wandb:       train/global_step 5625
wandb:         train/grad_norm 6.21744
wandb:          

wandb: 🚀 View run run-v2 at: https://wandb.ai/models-iit-jodhpur/mlops-assignment-group13/runs/6vbofn12
wandb: ⭐️ View project at: https://wandb.ai/models-iit-jodhpur/mlops-assignment-group13
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260611_164432-6vbofn12/logs



W&B run-v2 finished.


---
## Step 6: Compare Results

In [15]:
# Side-by-side comparison
print("=" * 60)
print("EXPERIMENT COMPARISON")
print("=" * 60)
print(f"{'Metric':<20} {'Version 1':<15} {'Version 2':<15}")
print("-" * 50)
print(f"{'Learning Rate':<20} {'2e-5':<15} {'5e-5':<15}")
print(f"{'Batch Size':<20} {'16':<15} {'32':<15}")
print(f"{'Epochs':<20} {'3':<15} {'3':<15}")
print("-" * 50)
print(f"{'Accuracy':<20} {results_v1['eval_accuracy']:<15.4f} {results_v2['eval_accuracy']:<15.4f}")
print(f"{'F1 (weighted)':<20} {results_v1['eval_f1']:<15.4f} {results_v2['eval_f1']:<15.4f}")
print(f"{'Eval Loss':<20} {results_v1['eval_loss']:<15.4f} {results_v2['eval_loss']:<15.4f}")
print("=" * 60)

# Determine best version
best = "v1" if results_v1['eval_f1'] > results_v2['eval_f1'] else "v2"
print(f"\nBest version: {best}")
print(f"\nCheck your W&B dashboard at: https://wandb.ai/ (project: mlops-assignment3)")

EXPERIMENT COMPARISON
Metric               Version 1       Version 2      
--------------------------------------------------
Learning Rate        2e-5            5e-5           
Batch Size           16              32             
Epochs               3               3              
--------------------------------------------------
Accuracy             0.9420          0.9412         
F1 (weighted)        0.9420          0.9412         
Eval Loss            0.4221          0.3680         

Best version: v1

Check your W&B dashboard at: https://wandb.ai/ (project: mlops-assignment3)


---
## Step 7: Push the Trained Model to Hugging Face Hub

Push the **best** trained model and tokenizer to a public Hugging Face repository so GitHub Actions can load it for inference.

> **Important**: Set the repository visibility to **Public** on Hugging Face before submitting.

In [16]:
# ---- CHANGE THIS to your Hugging Face username ----
HF_USERNAME = "Recurrent"
HF_REPO_NAME = "ag-news-distilbert"
HF_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"

# Select the best model based on F1 score
if results_v1["eval_f1"] >= results_v2["eval_f1"]:
    best_trainer = trainer_v1
    best_version = "v1"
else:
    best_trainer = trainer_v2
    best_version = "v2"

print(f"Best version: {best_version}")
print(f"Pushing to: https://huggingface.co/{HF_REPO_ID}")

Best version: v1
Pushing to: https://huggingface.co/Recurrent/ag-news-distilbert


In [17]:
# Push model and tokenizer to Hugging Face Hub
best_trainer.model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)

print(f"Model and tokenizer pushed to: https://huggingface.co/{HF_REPO_ID}")

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Model and tokenizer pushed to: https://huggingface.co/Recurrent/ag-news-distilbert


In [18]:
# Log the Hugging Face model URL into W&B run summary
hf_model_url = f"https://huggingface.co/{HF_REPO_ID}"

wandb.init(
    project="mlops-assignment-group13",
    name=f"push-model-{best_version}",
    config={"best_version": best_version, "hf_repo": HF_REPO_ID}
)
wandb.run.summary["huggingface_model"] = hf_model_url
wandb.run.summary["best_version"] = best_version
wandb.finish()


print(f"Logged HF model URL to W&B summary: {hf_model_url}")

wandb: setting up run w2v56pp4


wandb: Tracking run with wandb version 0.25.1


wandb: Run data is saved locally in /kaggle/working/wandb/run-20260611_172250-w2v56pp4
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run push-model-v1


wandb: ⭐️ View project at https://wandb.ai/models-iit-jodhpur/mlops-assignment-group13


...

wandb: uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml; uploading wandb-metadata.json


wandb: 
wandb: Run summary:
wandb:      best_version v1
wandb: huggingface_model https://huggingface....
wandb: 


wandb: 🚀 View run push-model-v1 at: https://wandb.ai/models-iit-jodhpur/mlops-assignment-group13/runs/w2v56pp4
wandb: ⭐️ View project at: https://wandb.ai/models-iit-jodhpur/mlops-assignment-group13
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260611_172250-w2v56pp4/logs


Logged HF model URL to W&B summary: https://huggingface.co/Recurrent/ag-news-distilbert


---
## Step 8: Push Notebook & Results to GitHub

Clone the repo, copy the notebook, commit and push to the `ritesh_dev` branch.

In [ ]:
import shutil
import os

# ── GitHub Config ──────────────────────────────────────────────────────────
GITHUB_USERNAME = "riteshmaury-iitj"
REPO_NAME = "group13-assignment-mlops"
BRANCH = "ritesh_dev"
REPO_PATH = f"/kaggle/working/{REPO_NAME}"

# ── Clone or update repo ──────────────────────────────────────────────────
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)
    print("Removed existing repo folder.")

!git clone https://{github_token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git {REPO_PATH}

# Checkout branch (create if it doesn't exist)
!cd {REPO_PATH} && git checkout {BRANCH} 2>/dev/null || git checkout -b {BRANCH}

# ── Configure git ──────────────────────────────────────────────────────────
!git -C {REPO_PATH} config user.email "ritesh@iitj.ac.in"
!git -C {REPO_PATH} config user.name "{GITHUB_USERNAME}"

# ── Copy notebook and files to repo ────────────────────────────────────────
import glob

# Copy notebooks
for nb in glob.glob("/kaggle/working/*.ipynb"):
    shutil.copy(nb, REPO_PATH)
    print(f"  Copied: {os.path.basename(nb)}")

# Copy id2label.json if it exists
for f in ["id2label.json"]:
    src = f"/kaggle/working/{f}"
    if os.path.exists(src):
        shutil.copy(src, REPO_PATH)
        print(f"  Copied: {f}")

# ── Commit and push ───────────────────────────────────────────────────────
!cd {REPO_PATH} && git add -A && git status
!cd {REPO_PATH} && git commit -m "Add training notebook with W&B tracking and HF model push"
!cd {REPO_PATH} && git push origin {BRANCH}

print(f"\nPushed to: https://github.com/{GITHUB_USERNAME}/{REPO_NAME}/tree/{BRANCH}")

---
## Notes for Report

1. **Screenshot**: Go to your W&B dashboard → project `mlops-assignment3` → select both runs → take a screenshot showing training loss, validation loss, accuracy, and F1 curves side by side.

2. **Hyperparameter differences**:
   - v1: lr=2e-5, batch_size=16
   - v2: lr=5e-5, batch_size=32

3. **All logged metrics**: training_loss, eval_loss, eval_accuracy, eval_f1 (logged every epoch via `eval_strategy='epoch'`)

4. **Kaggle Secrets used**: `WANDB_API_KEY`, `HF_TOKEN`, `GITHUB_TOKEN` — no tokens hardcoded in the notebook.

5. **Dataset source**: [Recurrent/prepared_data_mlops2](https://huggingface.co/datasets/Recurrent/prepared_data_mlops2)